In [1]:

import re
from dataclasses import dataclass
from typing import Dict, List, Set, Tuple, Optional
import pandas as pd

import sys
from pathlib import Path
sys.path.append("../")  # UI/
# sys.path.append(str(Path(__file__).parent.parent.parent))  # project root


In [2]:
parts_df = pd.read_csv("../data/ford-catalogue/merged_online_catalogue_annotated_parsed.csv")


parts_df = parts_df[parts_df["Annotated"] == True]
part_cols = [f'P{i}' for i in range(1, 6)]

def build_tokens_string(row):
    parts = []
    for col in part_cols:
        val = row.get(col)
        if pd.notna(val) and str(val).strip():
            parts.append(f"({str(val).strip()})")
    return ' '.join(parts) if parts else None

parts_df['tokens_string'] = parts_df.apply(build_tokens_string, axis=1)
parts_df[['filename', 'call_type', 'P1', 'P2', 'P3', 'P4', 'P5', 'tokens_string']]

DB = dict(zip(parts_df['filename'], parts_df['tokens_string']))


In [3]:
DB

{'N01i-A1-2': '(TIGHT FLAT) (NORMAL FLAT, TIGHT VALLEY) (PEAK, UP SLOW, PEAK)',
 'N01i-A1-3': '(BLUR) (TIGHT FLAT ?) (PEAK, FLAT)',
 'N01i-A1-4': '(TIGHT FLAT) (FLAT NORMAL) (PEAK, UP SLOW, PEAK)',
 'N01ii-B1-1': '(VERTICAL) (GAP) (PEAK, FLAT)',
 'N01ii-B1-2': '(VERTICAL) (GAP) (LEFT PEAK, DOWN SLOW, DOWN SHORT FAST)',
 'N01ii-I2-1': '(VERTICAL) (GAP) (PEAK LEFT, DOWN SLOW, DOWN SHORT FAST)',
 'N01iii-C1-1': '(VERTICAL) (TIGHT FLAT) (LEFT PEAK, FLAT)',
 'N01iii-D1-1': '(VERTICAL) (BLUR NOISY) (PEAK LEFT, FLAT)',
 'N01iv-H1-1': '(VERTICAL) (BLUR NOISY) (PEAK LEFT, DOWN)',
 'N01v-A4-1': '(BLUR) (BLUR NOISY) (LEFT PEAK, FLAT, UP SHORT FAST)',
 'N02-A1-2': '(UP NORMAL) (SQUIGGLE UP NORMAL) (UP FAST)',
 'N02-A4-1': '(BLUR) (SQUIGGLE, FLAT) (UP FAST SHORT)',
 'N02-A5-2': '(TIGHT FLAT) (UP SQUIGGLE) (UP SHORT FAST)',
 'N03-A1-2': '(BLUR) (TIGHT DOWN)',
 'N03-A5-1': '(BLUR) (TIGHT FLAT SHORT) (LEFT PEAK, DOWN TIGHT)',
 'N03-C1-1': '(BLUR) (GAP) (DOWN FAST TIGHT, DOWN TIGHT)',
 'N04-A1-1': '(LE

In [5]:
from token_library.call_query import build_index, search_calls

# ---- run demo ----
index = build_index(DB)

# queries = [
#     "ASC FAST PEAK FLAT",                 # should match N01i/N01ii-ish
#     "UPSWEEP FLAT ASC SBI_INC",            # should match N07ii/N07iii-like
#     "BB SBI_TIGHT SQUIGGLE",               # should match N02
#     "DESC SBI_WIDE +BIPHO -GAP",           # should match N25 style; requires BIPHO, forbids GAP
#     "DOWNSWEEP FLAT",                      # should match N23ii
#     "ASC SBI_WIDE PEAK RIGHT LARGE",       # should match N16ii-ish
# ]
queries = [
    "SQUIGGLE"    # should match N16ii-ish
]

for q in queries:
    print("\nQUERY:", q)
    display(search_calls(q, index, topk=8))



QUERY: SQUIGGLE


,Call,Score,Repr,Matched,Missing
0,N47-A1-1,0.333333,(BLUR) (UP SQUIGGLE),SQUIGGLE,
1,N02-A1-2,0.250000,(UP NORMAL) (SQUIGGLE UP NORMAL) (UP FAST),SQUIGGLE,
2,N30-I11-1,0.250000,"(UP SPACED, FLAT SPACED) (SQUIGGLE)",SQUIGGLE,
3,N30-I31-1,0.250000,"(UP SPACED, FLAT SPACED) (SQUIGGLE)",SQUIGGLE,
4,"N08iv-I1,I2,I18-1",0.200000,(VERTICAL) (PEAK NORMAL) (TIGHT SQUIGGLE),SQUIGGLE,
5,N13-A1-2,0.200000,(BLUR) (NORMAL FLAT) (SQUIGGLE) (VALLEY),SQUIGGLE,
6,N02-A4-1,0.166667,"(BLUR) (SQUIGGLE, FLAT) (UP FAST SHORT)",SQUIGGLE,
7,N02-A5-2,0.166667,(TIGHT FLAT) (UP SQUIGGLE) (UP SHORT FAST),SQUIGGLE,
